# Atelier Python Session 6 : le traitement automatique des langues avec nltk

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-6/Atelier%20Python%20Session%206%20-%20NLP%20-%20nltk.ipynb)

Aujourd'hui, nous allons aborder le [NLP](https://fr.wikipedia.org/wiki/Traitement_automatique_des_langues) (*Natural
Language Processing*), ou TAL en français (traitement automatique des langues). Il s'agit d'un ensemble de techniques
qui permettent d'extraire des informations d'un contenu textuel : découper un texte en phrases et en mots, trouver la
nature grammaticale des mots, les ramener à leur forme de dictionnaire, repérer les noms de personnes et de lieux...
Autant d'outils précieux pour analyser des corpus en SHS !

Nous allons utiliser [`nltk`](https://www.nltk.org/) (*Natural Language Toolkit*), un package historique et très
pédagogique du TAL en Python. Il existe un deuxième notebook pour cette séance, qui présente
[spaCy](https://spacy.io/), un outil plus moderne, qui fonctionne notamment mieux en français.

Voici le plan : chaque étape prépare la suivante, comme dans une vraie chaîne de traitement (on parle de *pipeline*).
1. la **tokenisation** : découper le texte en phrases et en mots ;
2. le **nettoyage** : retirer les mots vides et la ponctuation, et compter les mots ;
3. l'**étiquetage morpho-syntaxique** (*POS tagging*) : trouver la nature grammaticale de chaque mot ;
4. la **lemmatisation** : ramener chaque mot à sa forme de base ;
5. la **reconnaissance d'entités nommées** (*NER*) : repérer les personnes, les lieux, les organisations...

## Préparation

`nltk` est déjà installé dans Colab. En revanche, il a besoin de **modèles** et de **ressources** (listes de mots,
modèles statistiques...), qu'il faut télécharger une fois avec `nltk.download()`. Nous les téléchargerons au fur et à
mesure.

Voici le texte que nous allons analyser (en anglais, car la plupart des outils de `nltk` ne fonctionnent qu'en
anglais). On peut aussi lire un texte depuis un fichier, comme en séance 3.

In [ ]:
import nltk

# Des strings écrits les uns à la suite des autres entre parenthèses sont collés en un seul :
texte = ("In December 1720, a great fire burned for almost a week in Rennes, the capital of Brittany. "
         "Much of the medieval city centre was destroyed, and thousands of people lost their homes. "
         "The engineer Isaac Robelin, and later the architect Jacques Gabriel, drew up plans to rebuild the city "
         "with wide, straight streets and elegant stone buildings. "
         "Today, visitors can still see the contrast between the old timber-framed houses "
         "and the classical squares of the eighteenth century.")

print(texte)

## 1. La tokenisation

La **tokenisation** consiste à découper un texte en unités, appelées **tokens** : des phrases, ou des mots. Cela semble
simple (il suffirait de couper sur les espaces avec `split()`, non ?), mais la langue est pleine de pièges : la
ponctuation collée aux mots, les abréviations (« M. Dupont » n'est pas une fin de phrase), les apostrophes...

`nltk` propose deux fonctions : `sent_tokenize()` pour découper en phrases (*sentences*) et `word_tokenize()` pour
découper en mots. Elles ont besoin d'un modèle, `punkt_tab`, qu'il faut télécharger :

In [ ]:
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize, sent_tokenize

phrases = sent_tokenize(texte)
mots = word_tokenize(texte)

print(len(phrases), "phrases :")
for phrase in phrases:
  print("-", phrase)

print(len(mots), "tokens :")
print(mots)

Comparez avec un simple `split()` : la ponctuation reste collée aux mots (`"1720,"`, `"Brittany."`), alors que
`word_tokenize()` en fait des tokens à part.

In [ ]:
print(texte.split())

Ces fonctions marchent aussi pour d'autres langues, dont le français, avec l'argument `language` :

In [ ]:
texte_fr = "M. Dupont est arrivé à Rennes hier. Il n'a pas encore visité le Parlement de Bretagne !"

print(sent_tokenize(texte_fr, language="french"))
print(word_tokenize(texte_fr, language="french"))

## 2. Le nettoyage : mots vides et ponctuation

Si l'on compte les mots d'un texte, les plus fréquents seront presque toujours « the », « of », « and » (ou « le »,
« de », « et » en français). Ces mots très fréquents mais peu porteurs de sens s'appellent des **mots vides**
(*stop words*). Pour de nombreuses analyses (trouver les thèmes d'un texte, par exemple), on les retire.

`nltk` fournit des listes de mots vides pour de nombreuses langues. Pour la ponctuation, on peut utiliser
`string.punctuation`, du package `string` de Python.

> 💡 On transforme ces listes en **sets** avec `set()`. Un set est une collection sans ordre et sans doublons, dans
> laquelle la recherche avec `in` est beaucoup plus rapide que dans une liste.

In [ ]:
nltk.download("stopwords")

from nltk.corpus import stopwords
import string

mots_vides = set(stopwords.words("english"))
ponctuation = set(string.punctuation)

print(len(mots_vides), "mots vides en anglais, par exemple :", list(mots_vides)[:15])
print(ponctuation)

# Il existe aussi une liste française :
print(stopwords.words("french")[:15])

On peut maintenant filtrer nos mots : on garde un mot seulement s'il n'est pas un mot vide (on le compare en minuscules,
car la liste est en minuscules), et s'il est composé uniquement de lettres (la méthode `isalpha()` élimine la
ponctuation et les nombres).

In [ ]:
mots_filtres = []
for mot in mots:
  if mot.lower() not in mots_vides and mot.isalpha():
    mots_filtres.append(mot)

print(mots_filtres)

### Compter les mots avec `FreqDist`

Nous avons vu en séance 3 comment compter les mots avec un dictionnaire. `nltk` propose un outil tout prêt :
`nltk.FreqDist()` (distribution de fréquences). Sa méthode `most_common(n)` renvoie les `n` mots les plus fréquents,
sous forme de liste de **tuples** `(mot, nombre)` (un tuple est comme une liste qu'on ne peut pas modifier).

In [ ]:
frequences = nltk.FreqDist(mots_filtres)
print(frequences.most_common(5))

for mot, nombre in frequences.most_common(5):
  print(mot, ":", nombre)

## 3. L'étiquetage morpho-syntaxique (POS tagging)

Le **POS tagging** (*Part-Of-Speech tagging*) consiste à attribuer à chaque mot sa **catégorie grammaticale** : nom,
verbe, adjectif, adverbe... C'est plus difficile qu'il n'y paraît, car un même mot peut changer de catégorie selon le
contexte (« *the fire burned* » : nom ; « *they fire the cannon* » : verbe). Le modèle de `nltk` utilise donc les mots
voisins pour décider.

`nltk.pos_tag()` prend une liste de tokens et renvoie une liste de tuples `(mot, étiquette)`. Les étiquettes suivent les
conventions du [Penn Treebank](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html). Les plus
courantes :

| Étiquette | Catégorie | Exemple |
|---|---|---|
| `NN`, `NNS` | nom commun (singulier, pluriel) | *fire*, *streets* |
| `NNP` | nom propre | *Rennes* |
| `VB`, `VBD`, `VBN`... | verbe (base, passé, participe passé...) | *see*, *burned*, *destroyed* |
| `JJ` | adjectif | *great* |
| `RB` | adverbe | *still* |
| `DT` | déterminant | *the*, *a* |
| `IN` | préposition | *in*, *of* |

On étiquette le texte **complet** (avec les mots vides et la ponctuation) : le modèle a besoin du contexte de chaque mot
pour bien travailler.

In [ ]:
nltk.download("averaged_perceptron_tagger_eng")

etiquettes = nltk.pos_tag(mots)
print(etiquettes)

On peut par exemple ne garder que les noms (les étiquettes qui commencent par `NN`) :

In [ ]:
noms = []
for mot, etiquette in etiquettes:
  if etiquette.startswith("NN"):
    noms.append(mot)

print(noms)

## 4. La lemmatisation

La **lemmatisation** consiste à ramener chaque mot à son **lemme**, c'est-à-dire sa forme de dictionnaire : l'infinitif
pour un verbe (*burned* → *burn*), le singulier pour un nom (*streets* → *street*). C'est très utile pour compter les
occurrences d'un mot, quelle que soit sa forme.

`nltk` utilise pour cela [WordNet](https://wordnet.princeton.edu/), un grand dictionnaire de l'anglais. Mais le
lemmatiseur a besoin de connaître la catégorie grammaticale du mot : sans elle, il considère que tout est un nom, et ne
touche pas aux verbes. C'est là que le POS tagging devient utile !

In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.stem import WordNetLemmatizer

lemmatiseur = WordNetLemmatizer()

print(lemmatiseur.lemmatize("streets"))           # un nom : pas besoin de préciser
print(lemmatiseur.lemmatize("burned"))            # considéré comme un nom : rien ne change !
print(lemmatiseur.lemmatize("burned", pos="v"))   # en précisant que c'est un verbe : burn

Problème : les étiquettes du POS tagger (`VBD`, `NNS`...) ne sont pas dans le même format que celles attendues par
WordNet (`"v"`, `"n"`, `"a"`, `"r"`). Voici une fonction qui fait la traduction, en regardant la première lettre de
l'étiquette :

In [ ]:
from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):
  """Traduit une étiquette du Penn Treebank en catégorie WordNet."""
  if treebank_tag.startswith('J'):
    return wordnet.ADJ
  elif treebank_tag.startswith('V'):
    return wordnet.VERB
  elif treebank_tag.startswith('N'):
    return wordnet.NOUN
  elif treebank_tag.startswith('R'):
    return wordnet.ADV
  else:
    return wordnet.NOUN

print(get_wordnet_pos("VBD"))

On peut maintenant lemmatiser chaque mot de notre texte en lui donnant sa catégorie grammaticale. On en profite pour
retirer la ponctuation et les mots vides, et on range le résultat dans une liste de dictionnaires :

In [ ]:
lemmes = []

for mot, etiquette in etiquettes:
  if mot.isalpha() and mot.lower() not in mots_vides:
    lemme = lemmatiseur.lemmatize(mot.lower(), pos=get_wordnet_pos(etiquette))
    lemmes.append({"original": mot, "lemme": lemme})

for element in lemmes:
  if element["original"].lower() != element["lemme"]:
    print(element["original"], "->", element["lemme"])

## 5. La reconnaissance d'entités nommées (NER)

Les **entités nommées** sont les noms de personnes, de lieux, d'organisations, les dates... Les repérer
automatiquement (*Named Entity Recognition*) permet par exemple de savoir de qui et de quels lieux parle un corpus.

`nltk.ne_chunk()` prend la liste des tuples `(mot, étiquette)` du POS tagging, et renvoie un **arbre** (`nltk.Tree`)
dans lequel les mots qui forment une entité sont regroupés. Affichons-le :

In [ ]:
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

arbre = nltk.ne_chunk(etiquettes)
print(arbre)

Les entités apparaissent entre parenthèses, avec leur type : `PERSON` (personne), `GPE` (entité géopolitique : pays,
ville...), `ORGANIZATION`, etc.

Pour les récupérer dans une liste, on parcourt l'arbre avec une boucle `for`. Chaque élément est :
- soit un tuple `(mot, étiquette)` pour un mot ordinaire ;
- soit un sous-arbre (`nltk.Tree`) pour une entité. Sa méthode `.label()` donne le type d'entité, et `.leaves()` la liste
  des tuples `(mot, étiquette)` qui la composent.

In [ ]:
entites = []

for element in arbre:
  if type(element) == nltk.Tree:
    mots_entite = []
    for mot, etiquette in element.leaves():
      mots_entite.append(mot)
    # " ".join(liste) colle les strings de la liste en les séparant par des espaces :
    entites.append({"texte": " ".join(mots_entite), "type": element.label()})

for entite in entites:
  print(entite["texte"], ":", entite["type"])

Le résultat n'est pas parfait : certaines entités peuvent être manquées ou mal classées. Le modèle de `nltk` est ancien
et assez simple ; les outils plus récents, comme spaCy, font généralement mieux (et fonctionnent en français !).

## Pour aller plus loin

- Le deuxième notebook de cette séance, **Atelier Python Session 6 - NLP - spaCy**, refait tout ce pipeline avec spaCy,
  en anglais et en français, et présente l'analyse de sentiment et la modélisation thématique (*topic modelling*).
- Pour vous entraîner (avec vérification automatique et corrigé) : [Atelier Python Session 6 - Exercices.ipynb](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-6/Atelier%20Python%20Session%206%20-%20Exercices.ipynb).